In [1]:
from sage.all import *

class MatsumotoImai:
    """
    Matsumoto-Imai (C*) Kriptosisteminin adım adım izlenebilir
    implementasyonu.

    Matsumoto-Imai (MI) sistemi, Çok Değişkenli Polinom Tabanlı (ÇDPT)
    kriptografinin tarihsel olarak ilk önerilen yapılarından biridir
    ve "Big Field" (Büyük Cisim) ailesinin temelini oluşturur. Temel
    fikir, F_q^n uzayı ile F_{q^n} genişleme cismi arasındaki doğal
    izomorfizmadan yararlanarak, genişleme cismi üzerinde kolayca
    terslenebilen tek terimli (monomial) bir dönüşüm F(X) = X^(q^theta+1)
    tanımlamak; ardından bu dönüşümü iki rastgele afin dönüşüm (S ve T)
    ile "gizleyerek" görünüşte rastgele bir çok değişkenli kuadratik
    (MQ) polinom sistemi elde etmektir.

    Açık anahtar:
        P(x) = S( F( T(x) ) )
    burada:
        - T : F_q^n -> F_q^n gizli afin dönüşüm (giriş maskesi)
        - F : F_{q^n} -> F_{q^n} merkezi MI dönüşümü, F(X) = X^(q^theta+1)
        - S : F_q^n -> F_q^n gizli afin dönüşüm (çıkış maskesi)

    Terslenebilirlik koşulu:
        gcd(q^theta + 1, q^n - 1) = 1
    Bu koşul sağlandığında, F merkezi dönüşümünün tersi
    F^{-1}(Z) = Z^d biçiminde bir başka tek terimli (monomial)
    dönüşümdür; burada d, e = q^theta+1'in q^n-1 modülüne göre
    çarpımsal tersidir (e*d ≡ 1 mod (q^n - 1)). Ancak bu koşulun sağlanabilmesi için
    q çift olmalıdır aksi halde 2|gcd(q^theta + 1, q^n - 1) (böler)

    Referans: Tez, Bölüm 5 — BigField Tabanlı CDPT Kriptosistemleri
    """

    def __init__(self, q, n, theta):
        """
        Matsumoto-Imai (C*) Kriptosistemini verilen parametrelerle
        kurar ve terslenebilirlik koşulunu doğrular.

        Parametreler
        ------------
        q : int
            Temel cismin eleman sayısı (F_q). q ikinin
            kuvveti olmalıdır (q = 2^m).
        n : int
            Genişleme derecesi; aynı zamanda çok değişkenli polinom
            sistemindeki değişken sayısıdır. Genişleme cismi F_{q^n}
            olarak inşa edilir.
        theta : int
            MI merkezi dönüşümünün üs parametresi. Merkezi dönüşüm
            F(X) = X^(q^theta + 1) biçimindedir.

        Notlar
        ------
        Merkezi dönüşüm:
            F(X) = X^(q^theta + 1)

        Terslenebilirlik koşulu:
            gcd(q^theta + 1, q^n - 1) = 1

        Bu koşul sağlanmazsa yapıcı (constructor) bir ValueError
        fırlatır; çünkü bu durumda F merkezi dönüşümü F_{q^n}
        cisminin çarpımsal grubu üzerinde bire bir (injektif) değildir
        ve dolayısıyla şifre çözme işlemi tanımsız kalır.
        """
        self.q = q
        self.n = n
        self.theta = theta

        # ------------------------------------------------------------
        # 1. Temel Cisim ve Çok Değişkenli Polinom Halkası
        # ------------------------------------------------------------
        # F_q temel cismi ve bunun üzerinde n değişkenli polinom halkası
        # F_q[x_1, ..., x_n] kurulur. Açık anahtar bu halkada n adet
        # polinom olarak ifade edilecektir.
        self.F = GF(q, 'a')
        self.R = PolynomialRing(self.F, n, 'x')
        self.vars = vector(self.R, self.R.gens())

        # ------------------------------------------------------------
        # 2. Cisim Genişlemesi
        # ------------------------------------------------------------
        # F_q üzerinde derecesi n olan indirgenemez bir polinom seçilerek
        # F_{q^n} genişleme cismi E inşa edilir. MI sisteminin "Big Field"
        # yaklaşımının temeli budur: karmaşık MQ probleminin, E üzerinde
        # kolayca terslenebilen basit bir üs alma işlemine dönüştürülmesi.
        # Merkez dönüşüm ve ön görüntü alma işlemleri bu cisimde yapılır.
        self.R_uni = PolynomialRing(self.F, 'X')
        self.irr_poly = self.R_uni.irreducible_element(n)
        self.E = self.F.extension(self.irr_poly, 'w')

        print("\n" + "="*70)
        print("MATSUMOTO-IMAI (C*) SİSTEMİ")
        print(f"Parametreler: GF({q}), n={n}, theta={theta}")
        print(f"Genişleme Polinomu: {self.irr_poly}")
        print("="*70)

        # ------------------------------------------------------------
        # 3. MI Üsleri ve Terslenebilirlik Kontrolü
        # ------------------------------------------------------------
        # F(X) = X^(1 + q^theta)
        # F_{q^n}^* çarpımsal grubu (q^n - 1) mertebesindedir. F(X) = X^e
        # dönüşümünün bu grup üzerinde bire bir (dolayısıyla terslenebilir)
        # olması için e üssünün grup mertebesiyle aralarında asal olması
        # gerekir: gcd(e, q^n - 1) = 1. Bu koşul sağlanırsa, e'nin
        # (q^n-1) modülüne göre çarpımsal tersi d, F'in tersini
        # F^{-1}(Z) = Z^d biçiminde verir.
        self.e_exponent = 1 + q**theta
        self.modulus = q**n - 1

        g = gcd(ZZ(self.e_exponent), ZZ(self.modulus))

        print("\n[KURULUM] Merkezi harita üssü:")
        print(f"   e = 1 + q^theta = 1 + {q}^{theta} = {self.e_exponent}")
        print(f"   Modül = q^n - 1 = {q}^{n} - 1 = {self.modulus}")
        print(f"   gcd(e, q^n - 1) = {g}")

        if g != 1:
            raise ValueError(
                "HATA: gcd(1+q^theta, q^n-1) != 1. "
                "Bu parametrelerle MI merkez dönüşümü terslenebilir değildir."
            )

        # d, e'nin (q^n - 1) modülüne göre çarpımsal tersidir:
        # e * d ≡ 1 (mod q^n - 1). Bu değer, şifre çözme aşamasında
        # merkezi dönüşümün tersini almak için kullanılacaktır.
        self.d_exponent = inverse_mod(self.e_exponent, self.modulus)

        print("\n[KURULUM] Ters üs:")
        print(f"   d = {self.d_exponent}")
        print("   e*d ≡ 1 mod (q^n - 1)")

        # ------------------------------------------------------------
        # 4. Anahtar Bileşenleri
        # ------------------------------------------------------------
        # Gizli afin dönüşümler (S, T) ve açık anahtar, generate_keys()
        # çağrılana kadar tanımsızdır. Bu ön tanımlama, encrypt/decrypt
        # işlemlerinin anahtar üretilmeden çağrılmasını engellemeye
        # yarayan _check_keys_generated() kontrolü için gereklidir.
        self.S_mat = None
        self.S_vec = None
        self.T_mat = None
        self.T_vec = None
        self.Public_Key = None

    # ----------------------------------------------------------------
    # YARDIMCI KONTROL
    # ----------------------------------------------------------------
    def _check_keys_generated(self):
        """
        Açık/gizli anahtarların üretilip üretilmediğini kontrol eder.

        encrypt() ve decrypt() metotları, S ve T afin dönüşümlerine
        ve Public_Key polinom vektörüne ihtiyaç duyar. Bu metot, bu
        bileşenler henüz generate_keys() ile oluşturulmadan şifreleme
        veya şifre çözme denemesi yapıldığında anlaşılır bir hata
        mesajıyla erken uyarı verir.

        Yükseltir
        ---------
        ValueError
            Public_Key henüz üretilmemişse.
        """
        if self.Public_Key is None:
            raise ValueError("Önce generate_keys() metodu ile anahtar üretmelisiniz.")

    # ----------------------------------------------------------------
    # YARDIMCI DÖNÜŞÜMLER
    # ----------------------------------------------------------------
    def extension_to_vector(self, element):
        """
        Genişleme cismi E = F_{q^n} elemanını, F_q^n vektör uzayının
        bir elemanına dönüştürür (phi^{-1} işlemi).

        E, F_q üzerinde n boyutlu bir vektör uzayı olarak da
        görülebilir; E'nin bir elemanı, seçilen tabana (bu
        implementasyonda w = E.gen() kuvvetleri) göre F_q
        katsayılarından oluşan bir n-uzunluğunda vektöre karşılık
        gelir. Bu metot, merkezi dönüşümün E üzerinde uygulanmasından
        sonra sonucu tekrar F_q^n uzayına taşımak için kullanılır.

        Parametreler
        ------------
        element : E (F_{q^n}) elemanı
            F_q^n vektörüne dönüştürülecek genişleme cismi elemanı.

        Döndürür
        --------
        vector (F_q üzerinde, uzunluk n)
            element'in F_q üzerindeki katsayı vektörü.
        """
        try:
            poly = element.lift()
        except:
            try:
                poly = element.polynomial()
            except:
                poly = element

        poly = self.R_uni(poly)
        coeffs = poly.list()

        # Katsayı listesi n'den kısa olabilir (yüksek dereceli
        # terimlerin katsayısı sıfırsa); eksik konumlar sıfırla doldurulur
        while len(coeffs) < self.n:
            coeffs.append(self.F(0))

        return vector(self.F, coeffs)

    def vector_to_extension(self, vec):
        """
        F_q^n vektör uzayının bir elemanını, genişleme cismi
        E = F_{q^n}'nin bir elemanına dönüştürür (phi işlemi).

        Bu, extension_to_vector() metodunun tersidir: vec vektörü,
        E'nin w = E.gen() üreteci kuvvetleri tabanındaki katsayılar
        olarak yorumlanarak tek bir E elemanı elde edilir.

        Parametreler
        ------------
        vec : vector (F_q üzerinde, uzunluk n)
            Genişleme cismine taşınacak vektör.

        Döndürür
        --------
        E (F_{q^n}) elemanı
            vec'in temsil ettiği genişleme cismi elemanı.
        """
        w_gen = self.E.gen()
        res = self.E(0)

        for i in range(self.n):
            res += vec[i] * (w_gen**i)

        return res

    def reduce_F_ideal(self, polys):
        """
        Polinomları x_i^q - x_i = 0 cisim denklemleri idealine göre indirger.

        F_q sonlu cisminin her elemanı a, a^q = a (Frobenius sabit
        noktası) özelliğini sağlar. Bu nedenle x_i^q - x_i
        polinomları, F_q^n üzerindeki her nokta için sıfır değerini
        alır; bu polinomların oluşturduğu "cisim ideali" (field ideal)
        ile indirgeme yapmak, fonksiyonel olarak eşdeğer ama daha
        düşük dereceli (her değişkende en fazla derece q-1 olan)
        bir temsilci polinom elde etmeyi sağlar. Bu indirgeme, MI
        sisteminde açık anahtar polinomlarının kuadratik (derece 2)
        formda kalmasını garanti eder.

        Parametreler
        ------------
        polys : iterable of F_q[x_1,...,x_n] polinomları
            İndirgenecek polinomların listesi/vektörü.

        Döndürür
        --------
        vector (F_q[x_1,...,x_n) üzerinde)
            Her polinomun cisim idealine göre indirgenmiş hâli.
        """
        field_eqs = [self.vars[i]**self.q - self.vars[i] for i in range(self.n)]
        FieldIdeal = self.R.ideal(field_eqs)

        return vector(self.R, [FieldIdeal.reduce(p) for p in polys])

    # ----------------------------------------------------------------
    # ADIM 1: ANAHTAR ÜRETİMİ
    # ----------------------------------------------------------------
    def generate_keys(self):
        """
        MI kriptosisteminin açık ve gizli anahtarlarını üretir.

        İş akışı:
          1.1 İki rastgele terslenebilir afin dönüşüm (S, T) üretilir.
              Bunlar sistemin gizli anahtarını oluşturur.
          1.2 Açık anahtar P(x) = S(F(T(x))) formülüne göre hesaplanır:
              (A) y = T(x)               — giriş afin dönüşümü
              (B) Y = phi(y)              — F_q^n'den E = F_{q^n}'ye taşıma
              (C) Z = Y^(q^theta + 1)     — merkezi MI dönüşümü
              (D) Z_red = Z mod (cisim idealı) — kuadratik forma indirgeme
              (E) z = phi^{-1}(Z_red)     — E'den F_q^n'ye geri taşıma
              (F) P = S(z) = S_mat*z + S_vec — çıkış afin dönüşümü

        Döndürür
        --------
        vector (F_q[x_1,...,x_n] üzerinde, uzunluk n)
            Açık anahtarı oluşturan n adet çok değişkenli kuadratik
            polinom. Aynı zamanda self.Public_Key olarak saklanır.
        """
        print("\n" + "*"*30 + " ADIM 1: ANAHTAR ÜRETİMİ " + "*"*30)

        # ------------------------------------------------------------
        # 1.1 Gizli Afin Dönüşümler S ve T
        # ------------------------------------------------------------
        # T ve S, F_q^n uzayı üzerinde terslenebilir afin dönüşümlerdir
        # (T(x) = T_mat*x + T_vec, S(x) = S_mat*x + S_vec). Terslenebilir
        # olmaları, decrypt() aşamasında bu dönüşümlerin tersinin
        # alınabilmesi için zorunludur; bu nedenle matrisler tekrar tekrar
        # örneklenerek determinantın sıfırdan farklı olduğu (is_invertible())
        # durum elde edilene kadar denenir.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible():
                break

        self.T_vec = random_vector(self.F, self.n)

        while True:
            self.S_mat = random_matrix(self.F, self.n, self.n)
            if self.S_mat.is_invertible():
                break

        self.S_vec = random_vector(self.F, self.n)

        print("\n[1.1] Gizli Afin Dönüşümler:")
        print(f"   T Matrisi (Giriş dönüşümü):\n{self.T_mat}")
        print(f"   S Matrisi (Çıkış dönüşümü):\n{self.S_mat}")
        print(f"   Sabit Vektör c_T: {self.T_vec}")
        print(f"   Sabit Vektör c_S: {self.S_vec}")

        # ------------------------------------------------------------
        # 1.2 Açık Anahtar Hesabı
        # ------------------------------------------------------------
        print("\n[1.2] Açık Anahtar Hesaplanıyor:")
        print("      P(x) = S( F( T(x) ) )")

        # A) T(x) -> y : gizli giriş dönüşümü uygulanır
        y_vec = self.T_mat * self.vars + self.T_vec
        print(f"\n   -> y = T(x) hesaplandı:")
        print(f"      y(x) = {y_vec}")

        # B) Genişleme cismine taşıma: y vektörünün her bileşeni,
        # E = F_{q^n} üzerinde tanımlı polinom halkasına gömülür ve
        # phi izomorfizması ile tek bir Y elemanı (E katsayılı polinom
        # olarak) oluşturulur.
        R_E_multi = PolynomialRing(self.E, self.n, 'x')
        y_vec_E = vector(R_E_multi, [R_E_multi(p) for p in y_vec])

        w_gen = self.E.gen()
        Y_sym = sum(y_vec_E[i] * (w_gen**i) for i in range(self.n))

        print("\n   -> y vektörü genişleme cismine taşındı:")
        print(f"      Y = phi(y) = {Y_sym}")

        # C) Merkezi MI haritası: Y, E üzerinde tanımlı basit bir
        # tek terimli (monomial) dönüşüme, yani üs almaya tabi tutulur.
        # Bu adım, F_{q^n} cisminin cebirsel yapısı sayesinde E
        # üzerinde kolayca hesaplanabilir; ancak F_q^n'ye geri
        # indirgendiğinde görünüşte rastgele bir kuadratik sistem gibi
        # görünür.
        Z_sym = Y_sym ** self.e_exponent

        print("\n   -> Merkezi MI haritası uygulandı:")
        print(f"      Z = Y^{self.e_exponent}")

        # D) Cisim denklemleriyle indirgeme: E katsayılı çok değişkenli
        # polinom halkasında x_i^q - x_i = 0 idealine göre indirgeme
        # yapılarak Z_sym'in derecesi düşürülür. Bu adımın MI sisteminde
        # kritik önemi vardır: indirgeme sonrası elde edilen polinomun
        # her değişkende en fazla derece q-1 olması ve toplam derecenin
        # 2'ye inmesi beklenir (bu, açık anahtarın neden "kuadratik"
        # bir MQ sistemi olduğunu açıklar).
        print(f"\n   -> Cisim denklemleriyle indirgeme yapılıyor:")
        print(f"      x_i^{self.q} - x_i = 0")

        vars_E = R_E_multi.gens()
        field_eqs_E = [vars_E[i]**self.q - vars_E[i] for i in range(self.n)]
        Ideal_E = R_E_multi.ideal(field_eqs_E)

        Z_reduced = Ideal_E.reduce(Z_sym)

        print(f"      İndirgenmiş derece: {Z_reduced.degree()} (MI için beklenen derece: 2)")
        print(f"      Z_reduced = {Z_reduced}")

        # E) Genişleme cisminden F_q^n vektörüne dönüş: Z_reduced
        # polinomunun her bir monomunun E-katsayısı, phi^{-1} ile
        # F_q^n vektörüne çevrilir ve bu vektörün k. bileşeni,
        # z_polys[k] polinomuna ilgili monomiyal terim olarak eklenir.
        # Sonuçta z_vec, F_q[x_1,...,x_n] halkasında n adet polinomdan
        # oluşan bir vektördür.
        z_polys = [self.R(0) for _ in range(self.n)]

        for exps, coeff in Z_reduced.dict().items():
            coeff_vec = self.extension_to_vector(coeff)
            monomial = self.R.monomial(*exps)

            for k in range(self.n):
                z_polys[k] += coeff_vec[k] * monomial

        z_vec = vector(self.R, z_polys)

        print("\n   -> phi^{-1} sonrası ara vektör:")
        print(f"      z(x) = {z_vec}")

        # F) S dönüşümü: gizli çıkış afin dönüşümü uygulanarak açık
        # anahtar tamamlanır. S'nin varlığı, z(x)'in cebirsel yapısının
        # (örneğin merkezi haritadan miras kalan özel örüntülerin)
        # dışarıdan görünmesini engellemeye yarar.
        self.Public_Key = self.S_mat * z_vec + self.S_vec

        # Temiz temsil için son indirgeme: açık anahtar polinomları da
        # cisim idealine göre indirgenerek en sade (kanonik) forma
        # getirilir.
        self.Public_Key = self.reduce_F_ideal(self.Public_Key)

        print("\n[1.3] AÇIK ANAHTAR P_pub:")
        for idx, p in enumerate(self.Public_Key):
            print(f"   P_{idx} = {p}")

        return self.Public_Key

    # ----------------------------------------------------------------
    # ADIM 2: AÇIK ANAHTAR DÖNÜŞÜMÜ
    # ----------------------------------------------------------------
    def encrypt(self, message_vec):
        """
        Açık anahtar dönüşümünü uygular.

        Kriptosistem diliyle:
            y = P_pub(x)

        Tez diliyle:
            Açık anahtar haritası uygulanır.

        Bu işlem, açık anahtarı oluşturan n adet çok değişkenli
        polinomun her birine message_vec değerlerinin (x_1,...,x_n
        değişkenleri yerine) yerleştirilmesinden (substitution)
        ibarettir; gizli S, T dönüşümleri veya merkezi MI haritası
        bu aşamada doğrudan kullanılmaz — tüm bu bilgi zaten
        Public_Key polinomlarının içine gömülmüştür.

        Parametreler
        ------------
        message_vec : vector (F_q üzerinde, uzunluk n)
            Şifrelenecek/dönüştürülecek girdi vektörü (düz metin).

        Döndürür
        --------
        vector (F_q üzerinde, uzunluk n)
            Açık anahtar polinomlarının message_vec'te değerlendirilmesiyle
            elde edilen çıktı vektörü (şifreli metin).
        """
        self._check_keys_generated()

        print("\n" + "*"*30 + " ADIM 2: AÇIK ANAHTAR UYGULAMA " + "*"*30)
        print(f"   Girdi vektörü (x): {message_vec}")

        val_dict = {self.vars[i]: message_vec[i] for i in range(self.n)}
        ciphertext = vector(self.F, [p.subs(val_dict) for p in self.Public_Key])

        print(f"   Açık anahtar çıktısı (y): {ciphertext}")

        return ciphertext

    # ----------------------------------------------------------------
    # ADIM 3: GİZLİ ANAHTARLA TERSLEME
    # ----------------------------------------------------------------
    def decrypt(self, ciphertext_vec):
        """
        Gizli anahtar ile açık anahtar çıktısının ön görüntüsünü bulur.

        Kriptosistem diliyle:
            x = P_pub^{-1}(y)

        Tez diliyle:
            Gizli yapı kullanılarak ön görüntü bulunur.

        İş akışı, generate_keys() içindeki P(x) = S(F(T(x)))
        yapısının tam tersi sırayla uygulanmasından oluşur:
          3.1 S^{-1}    : çıkış afin dönüşümünün tersi
          3.2 phi       : F_q^n'den E = F_{q^n}'ye taşıma
          3.3 F^{-1}    : merkezi MI dönüşümünün tersi (üs alma ile)
          3.4 phi^{-1}  : E'den F_q^n'ye geri taşıma
          3.5 T^{-1}    : giriş afin dönüşümünün tersi

        Bu tersleme yalnızca S, T ve d_exponent (merkezi haritanın
        ters üssü) bilgisine sahip taraf tarafından verimli şekilde
        gerçekleştirilebilir; bu bilgiler kriptosistemin gizli
        anahtarını oluşturur.

        Parametreler
        ------------
        ciphertext_vec : vector (F_q üzerinde, uzunluk n)
            Ön görüntüsü bulunacak hedef vektör (şifreli metin).

        Döndürür
        --------
        vector (F_q üzerinde, uzunluk n)
            P_pub(x) = ciphertext_vec denklemini sağlayan x vektörü
            (geri kazanılan düz metin).
        """
        self._check_keys_generated()

        print("\n" + "*"*30 + " ADIM 3: GİZLİ ANAHTARLA TERSLEME " + "*"*30)
        print(f"   Hedef vektör (y): {ciphertext_vec}")

        # 1. S^{-1}: gizli çıkış dönüşümünün tersi uygulanarak
        # S(z) = ciphertext_vec denkleminden z = w elde edilir
        w_vec = self.S_mat.inverse() * (ciphertext_vec - self.S_vec)

        print("\n[3.1] S^{-1} uygulandı:")
        print(f"      w = {w_vec}")

        # 2. phi: w vektörü genişleme cismine taşınarak tek bir
        # E elemanı W elde edilir
        W = self.vector_to_extension(w_vec)

        print("\n[3.2] Genişleme cismine geçildi:")
        print(f"      W = phi(w) = {W}")

        # 3. Merkezi harita tersi: F(Y) = Y^e dönüşümünün tersi,
        # F_{q^n}^* çarpımsal grubunda basitçe d = e^{-1} mod (q^n-1)
        # üssüne yükseltmektir; bu sayede Y = W^d hesaplanarak
        # merkezi dönüşümden önceki değer geri elde edilir.
        R_elem = W ** self.d_exponent

        print(f"\n[3.3] Merkezi harita tersi uygulandı:")
        print(f"      R = W^d, d = {self.d_exponent}")
        print(f"      R = {R_elem}")

        # 4. phi^{-1}: E elemanı tekrar F_q^n vektörüne çevrilir
        r_vec = self.extension_to_vector(R_elem)

        print("\n[3.4] Temel cisme geri dönüldü:")
        print(f"      r = phi^(-1)(R) = {r_vec}")

        # 5. T^{-1}: gizli giriş dönüşümünün tersi uygulanarak
        # T(x) = r_vec denkleminden orijinal x geri kazanılır
        plaintext = self.T_mat.inverse() * (r_vec - self.T_vec)

        print("\n[3.5] T^{-1} uygulandı:")
        print(f"      x = {plaintext}")

        return plaintext

    # ----------------------------------------------------------------
    # TEZ NOTASYONUNA UYGUN ALIAS METOTLAR
    # ----------------------------------------------------------------
    def public_map(self, message_vec):
        """
        Açık anahtar dönüşümü P(x)'i uygular.

        encrypt() metodunun, tez içindeki P(x) notasyonuna daha uygun
        isimlendirilmiş bir takma adıdır (alias); işlevsel olarak
        encrypt() ile birebir aynıdır.

        Parametreler
        ------------
        message_vec : vector (F_q üzerinde, uzunluk n)
            Açık anahtar haritasının uygulanacağı girdi vektörü.

        Döndürür
        --------
        vector (F_q üzerinde, uzunluk n)
            P(message_vec) değeri.
        """
        return self.encrypt(message_vec)

    def invert_with_secret_key(self, target_vec):
        """
        Gizli anahtar ile P(x) = target_vec ön görüntüsünü bulur.

        decrypt() metodunun, tez içindeki gizli anahtarla tersleme
        işlemine daha uygun isimlendirilmiş bir takma adıdır (alias);
        işlevsel olarak decrypt() ile birebir aynıdır.

        Parametreler
        ------------
        target_vec : vector (F_q üzerinde, uzunluk n)
            Ön görüntüsü aranan hedef vektör.

        Döndürür
        --------
        vector (F_q üzerinde, uzunluk n)
            P(x) = target_vec denklemini sağlayan x vektörü.
        """
        return self.decrypt(target_vec)



In [2]:

# ============================================================
# SENARYO: MI sistemi kurulumu ve tersleme testi
# ============================================================
# GF(4) temel cismi, n=3 genişleme derecesi ve theta=2 MI parametresi
# ile bir Matsumoto-Imai sistemi kurulur. Bu senaryo, anahtar üretimi,
# açık anahtar ile şifreleme (public_map) ve gizli anahtar ile
# şifre çözmenin (invert_with_secret_key) uçtan uca doğru çalıştığını,
# yani P^{-1}(P(x)) = x özdeşliğinin sağlandığını doğrulamayı amaçlar.

q = 4
n = 3
theta = 2

mi = MatsumotoImai(q, n, theta)

# Anahtar üretimi
P = mi.generate_keys()

# Rastgele girdi vektörü
msg = random_vector(mi.F, mi.n)

print("\nRastgele seçilen girdi vektörü:")
print(f"x = {msg}")

# Açık anahtar dönüşümü
y = mi.public_map(msg)

# Gizli anahtarla ön görüntü bulma
x_recovered = mi.invert_with_secret_key(y)

print("\n" + "="*70)
print("SONUÇ KONTROLÜ")
print("="*70)
print(f"Başlangıç vektörü     : {msg}")
print(f"Geri kazanılan vektör : {x_recovered}")

if msg == x_recovered:
    print("SAĞLAMA: BAŞARILI")
else:
    print("SAĞLAMA: HATA")
print("="*70)


MATSUMOTO-IMAI (C*) SİSTEMİ
Parametreler: GF(4), n=3, theta=2
Genişleme Polinomu: X^3 + a

[KURULUM] Merkezi harita üssü:
   e = 1 + q^theta = 1 + 4^2 = 17
   Modül = q^n - 1 = 4^3 - 1 = 63
   gcd(e, q^n - 1) = 1

[KURULUM] Ters üs:
   d = 26
   e*d ≡ 1 mod (q^n - 1)

****************************** ADIM 1: ANAHTAR ÜRETİMİ ******************************

[1.1] Gizli Afin Dönüşümler:
   T Matrisi (Giriş dönüşümü):
[    a     1     a]
[    1 a + 1 a + 1]
[    a     0 a + 1]
   S Matrisi (Çıkış dönüşümü):
[    a     a a + 1]
[    0 a + 1     a]
[    a     a     a]
   Sabit Vektör c_T: (0, 1, 1)
   Sabit Vektör c_S: (a, a, a + 1)

[1.2] Açık Anahtar Hesaplanıyor:
      P(x) = S( F( T(x) ) )

   -> y = T(x) hesaplandı:
      y(x) = ((a)*x0 + x1 + (a)*x2, x0 + (a + 1)*x1 + (a + 1)*x2 + 1, (a)*x0 + (a + 1)*x2 + 1)

   -> y vektörü genişleme cismine taşındı:
      Y = phi(y) = (a*w^2 + w + a)*x0 + ((a + 1)*w + 1)*x1 + ((a + 1)*w^2 + (a + 1)*w + a)*x2 + w^2 + w

   -> Merkezi MI haritası uygula